## Imports

In [1]:
from __future__ import annotations

import gc
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import timesfm
import wandb


CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

TABLES_DIR = (
    PROJECT_ROOT
    / "reports"
    / "tables"
)

SUBMISSIONS_DIR = (
    PROJECT_ROOT
    / "reports"
    / "submissions"
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SUBMISSIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.set_float32_matmul_precision(
    "high"
)

pd.set_option(
    "display.max_columns",
    100,
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "TimesFM package:",
    version("timesfm"),
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "CUDA available:",
    torch.cuda.is_available(),
)

/home/xizusha/Documents/ML/walmart-store-sales-forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /home/xizusha/Documents/ML/walmart-store-sales-forecasting
TimesFM package: 2.0.2
PyTorch: 2.13.0+cu130
CUDA available: True


#

In [2]:
SEED = 42

ENTITY = (
    "lkhiz23-free-university-of-tbilisi-"
)

WANDB_PROJECT = (
    "walmart-store-sales-forecasting"
)

MODEL_ID = (
    "google/timesfm-2.5-200m-pytorch"
)

MODEL_VERSION = "2.5"
TRAINING_MODE = "zero_shot"

CV_FOLDS = 3
CV_HORIZON = 13
HOLDOUT_HORIZON = 39

MAX_CONTEXT = 256
MIN_CONTEXT_POINTS = 32
MIN_CONTEXT_COVERAGE = 0.80

FORECAST_CHUNK_SIZE = 256

PER_CORE_BATCH_SIZE = (
    64
    if torch.cuda.is_available()
    else 16
)

RUN_NAME = (
    "timesfm__evaluation__"
    "zero_shot_target_only_v1__s42"
)

np.random.seed(SEED)
torch.manual_seed(SEED)

print(
    "Model:",
    MODEL_ID,
)

print(
    "Batch size:",
    PER_CORE_BATCH_SIZE,
)

Model: google/timesfm-2.5-200m-pytorch
Batch size: 64


## Load data

In [3]:
train_path = (
    RAW_DATA_DIR
    / "train.csv"
)

test_path = (
    RAW_DATA_DIR
    / "test.csv"
)

if not train_path.exists():
    raise FileNotFoundError(
        f"Missing train file: {train_path}"
    )

if not test_path.exists():
    raise FileNotFoundError(
        f"Missing test file: {test_path}"
    )

train = pd.read_csv(
    train_path,
    parse_dates=["Date"],
)

test = pd.read_csv(
    test_path,
    parse_dates=["Date"],
)

required_train_columns = {
    "Store",
    "Dept",
    "Date",
    "Weekly_Sales",
    "IsHoliday",
}

required_test_columns = {
    "Store",
    "Dept",
    "Date",
    "IsHoliday",
}

missing_train_columns = (
    required_train_columns
    - set(train.columns)
)

missing_test_columns = (
    required_test_columns
    - set(test.columns)
)

if missing_train_columns:
    raise ValueError(
        "Train is missing columns: "
        f"{sorted(missing_train_columns)}"
    )

if missing_test_columns:
    raise ValueError(
        "Test is missing columns: "
        f"{sorted(missing_test_columns)}"
    )

if train.duplicated(
    ["Store", "Dept", "Date"]
).any():
    raise ValueError(
        "Duplicate training rows detected."
    )

if test.duplicated(
    ["Store", "Dept", "Date"]
).any():
    raise ValueError(
        "Duplicate test rows detected."
    )

train = train.sort_values(
    ["Date", "Store", "Dept"]
).reset_index(drop=True)

test = test.reset_index(drop=True)

print(
    "Train rows:",
    len(train),
)

print(
    "Train dates:",
    train["Date"].nunique(),
)

print(
    "Train range:",
    train["Date"].min(),
    "—",
    train["Date"].max(),
)

print(
    "Test rows:",
    len(test),
)

print(
    "Test dates:",
    test["Date"].nunique(),
)

print(
    "Test range:",
    test["Date"].min(),
    "—",
    test["Date"].max(),
)

Train rows: 421570
Train dates: 143
Train range: 2010-02-05 00:00:00 — 2012-10-26 00:00:00
Test rows: 115064
Test dates: 39
Test range: 2012-11-02 00:00:00 — 2013-07-26 00:00:00


## Temporal validation splits

In [4]:
all_dates = pd.DatetimeIndex(
    train["Date"]
    .drop_duplicates()
    .sort_values()
)

if len(all_dates) <= HOLDOUT_HORIZON:
    raise ValueError(
        "Not enough dates for the "
        "39-week final holdout."
    )

development_dates = (
    all_dates[
        :-HOLDOUT_HORIZON
    ]
)

holdout_dates = (
    all_dates[
        -HOLDOUT_HORIZON:
    ]
)

required_cv_dates = (
    CV_FOLDS
    * CV_HORIZON
)

if (
    len(development_dates)
    <= required_cv_dates
):
    raise ValueError(
        "Not enough development dates "
        "for expanding CV."
    )

cv_splits = []

first_validation_index = (
    len(development_dates)
    - required_cv_dates
)

for fold_index in range(CV_FOLDS):
    validation_start = (
        first_validation_index
        + fold_index
        * CV_HORIZON
    )

    validation_end = (
        validation_start
        + CV_HORIZON
    )

    fold_train_dates = (
        development_dates[
            :validation_start
        ]
    )

    fold_validation_dates = (
        development_dates[
            validation_start:
            validation_end
        ]
    )

    cv_splits.append(
        {
            "fold": fold_index + 1,
            "train_dates": (
                fold_train_dates
            ),
            "validation_dates": (
                fold_validation_dates
            ),
        }
    )

for split in cv_splits:
    print(
        f"Fold {split['fold']}:",
        f"train through "
        f"{split['train_dates'].max().date()},",
        f"validate "
        f"{split['validation_dates'].min().date()}",
        "to",
        split[
            "validation_dates"
        ].max().date(),
    )

print(
    "\nFinal holdout:",
    holdout_dates.min().date(),
    "to",
    holdout_dates.max().date(),
)

Fold 1: train through 2011-04-29, validate 2011-05-06 to 2011-07-29
Fold 2: train through 2011-07-29, validate 2011-08-05 to 2011-10-28
Fold 3: train through 2011-10-28, validate 2011-11-04 to 2012-01-27

Final holdout: 2012-02-03 to 2012-10-26


## Metrics

In [5]:
def weighted_mean_absolute_error(
    actual: np.ndarray,
    predicted: np.ndarray,
    is_holiday: np.ndarray,
) -> float:
    actual = np.asarray(
        actual,
        dtype=float,
    )

    predicted = np.asarray(
        predicted,
        dtype=float,
    )

    is_holiday = np.asarray(
        is_holiday,
        dtype=bool,
    )

    weights = np.where(
        is_holiday,
        5.0,
        1.0,
    )

    return float(
        np.sum(
            weights
            * np.abs(
                actual
                - predicted
            )
        )
        / np.sum(weights)
    )


def calculate_metrics(
    prediction_frame: pd.DataFrame,
) -> dict[str, float]:
    actual = prediction_frame[
        "Weekly_Sales"
    ].to_numpy(dtype=float)

    predicted = prediction_frame[
        "Prediction"
    ].to_numpy(dtype=float)

    is_holiday = prediction_frame[
        "IsHoliday"
    ].to_numpy(dtype=bool)

    absolute_error = np.abs(
        actual - predicted
    )

    squared_error = (
        actual - predicted
    ) ** 2

    holiday_mask = is_holiday
    ordinary_mask = ~is_holiday

    return {
        "wmae": (
            weighted_mean_absolute_error(
                actual=actual,
                predicted=predicted,
                is_holiday=is_holiday,
            )
        ),
        "mae": float(
            absolute_error.mean()
        ),
        "rmse": float(
            np.sqrt(
                squared_error.mean()
            )
        ),
        "holiday_mae": float(
            absolute_error[
                holiday_mask
            ].mean()
            if holiday_mask.any()
            else np.nan
        ),
        "non_holiday_mae": float(
            absolute_error[
                ordinary_mask
            ].mean()
            if ordinary_mask.any()
            else np.nan
        ),
    }

## Context and fallback preparation

In [6]:
def prepare_forecast_inputs(
    training_rows: pd.DataFrame,
    target_rows: pd.DataFrame,
) -> dict:
    training_rows = (
        training_rows.copy()
    )

    target_pairs = (
        target_rows[
            ["Store", "Dept"]
        ]
        .drop_duplicates()
        .sort_values(
            ["Store", "Dept"]
        )
        .reset_index(drop=True)
    )

    all_context_dates = (
        pd.date_range(
            start=training_rows[
                "Date"
            ].min(),
            end=training_rows[
                "Date"
            ].max(),
            freq="W-FRI",
        )
    )

    context_dates = (
        all_context_dates[
            -MAX_CONTEXT:
        ]
    )

    panel = (
        training_rows
        .pivot(
            index=[
                "Store",
                "Dept",
            ],
            columns="Date",
            values="Weekly_Sales",
        )
        .reindex(
            columns=context_dates
        )
    )

    eligible_keys = []
    model_inputs = []
    coverage_rows = []

    for row in target_pairs.itertuples(
        index=False
    ):
        key = (
            int(row.Store),
            int(row.Dept),
        )

        if key not in panel.index:
            coverage_rows.append(
                {
                    "Store": key[0],
                    "Dept": key[1],
                    "observed_points": 0,
                    "context_length": 0,
                    "coverage": 0.0,
                    "eligible": False,
                }
            )

            continue

        series = (
            panel
            .loc[key]
            .to_numpy(dtype=float)
        )

        observed_mask = (
            np.isfinite(series)
        )

        if not observed_mask.any():
            continue

        first_observed = int(
            np.flatnonzero(
                observed_mask
            )[0]
        )

        series = series[
            first_observed:
        ]

        observed_mask = observed_mask[
            first_observed:
        ]

        observed_points = int(
            observed_mask.sum()
        )

        coverage = float(
            observed_mask.mean()
        )

        eligible = (
            observed_points
            >= MIN_CONTEXT_POINTS
            and coverage
            >= MIN_CONTEXT_COVERAGE
        )

        coverage_rows.append(
            {
                "Store": key[0],
                "Dept": key[1],
                "observed_points": (
                    observed_points
                ),
                "context_length": (
                    len(series)
                ),
                "coverage": coverage,
                "eligible": eligible,
            }
        )

        if not eligible:
            continue

        filled_series = np.where(
            observed_mask,
            series,
            0.0,
        ).astype(np.float32)

        eligible_keys.append(key)
        model_inputs.append(
            filled_series
        )

    recent_training_dates = (
        training_rows[
            "Date"
        ]
        .drop_duplicates()
        .sort_values()
        .iloc[-52:]
    )

    recent_training = (
        training_rows[
            training_rows[
                "Date"
            ].isin(
                recent_training_dates
            )
        ]
    )

    if recent_training.empty:
        recent_training = training_rows

    pair_means = (
        recent_training
        .groupby(
            ["Store", "Dept"]
        )["Weekly_Sales"]
        .mean()
    )

    department_means = (
        recent_training
        .groupby("Dept")[
            "Weekly_Sales"
        ]
        .mean()
    )

    store_means = (
        recent_training
        .groupby("Store")[
            "Weekly_Sales"
        ]
        .mean()
    )

    global_mean = float(
        recent_training[
            "Weekly_Sales"
        ].mean()
    )

    return {
        "eligible_keys": eligible_keys,
        "model_inputs": model_inputs,
        "coverage_frame": (
            pd.DataFrame(
                coverage_rows
            )
        ),
        "pair_means": pair_means,
        "department_means": (
            department_means
        ),
        "store_means": store_means,
        "global_mean": global_mean,
    }

## TimesFM prediction function

In [7]:
def create_fallback_predictions(
    rows: pd.DataFrame,
    prepared_data: dict,
) -> np.ndarray:
    pair_index = (
        pd.MultiIndex.from_frame(
            rows[
                ["Store", "Dept"]
            ]
        )
    )

    fallback = pd.Series(
        prepared_data[
            "pair_means"
        ]
        .reindex(pair_index)
        .to_numpy(),
        index=rows.index,
        dtype=float,
    )

    fallback = fallback.fillna(
        rows["Dept"].map(
            prepared_data[
                "department_means"
            ]
        )
    )

    fallback = fallback.fillna(
        rows["Store"].map(
            prepared_data[
                "store_means"
            ]
        )
    )

    fallback = fallback.fillna(
        prepared_data[
            "global_mean"
        ]
    )

    return fallback.to_numpy(
        dtype=np.float32
    )


def forecast_target_rows(
    model,
    training_rows: pd.DataFrame,
    target_rows: pd.DataFrame,
    horizon: int,
) -> tuple[
    pd.DataFrame,
    dict[str, float],
]:
    target_rows = (
        target_rows.copy()
        .reset_index(drop=True)
    )

    target_rows[
        "_original_order"
    ] = np.arange(
        len(target_rows)
    )

    target_dates = pd.DatetimeIndex(
        target_rows[
            "Date"
        ]
        .drop_duplicates()
        .sort_values()
    )

    if len(target_dates) != horizon:
        raise ValueError(
            f"Expected {horizon} target "
            f"dates, found "
            f"{len(target_dates)}."
        )

    prepared_data = (
        prepare_forecast_inputs(
            training_rows=training_rows,
            target_rows=target_rows,
        )
    )

    eligible_keys = (
        prepared_data[
            "eligible_keys"
        ]
    )

    model_inputs = (
        prepared_data[
            "model_inputs"
        ]
    )

    forecast_started = time.perf_counter()

    point_forecasts = []

    for chunk_start in range(
        0,
        len(model_inputs),
        FORECAST_CHUNK_SIZE,
    ):
        chunk_inputs = model_inputs[
            chunk_start:
            chunk_start
            + FORECAST_CHUNK_SIZE
        ]

        chunk_point, _ = (
            model.forecast(
                horizon=horizon,
                inputs=chunk_inputs,
            )
        )

        chunk_point = np.asarray(
            chunk_point,
            dtype=np.float32,
        )

        if not np.isfinite(
            chunk_point
        ).all():
            raise FloatingPointError(
                "TimesFM returned "
                "non-finite values."
            )

        point_forecasts.append(
            chunk_point
        )

    predict_seconds = (
        time.perf_counter()
        - forecast_started
    )

    if point_forecasts:
        point_forecasts = (
            np.concatenate(
                point_forecasts,
                axis=0,
            )
        )

        key_frame = pd.DataFrame(
            eligible_keys,
            columns=[
                "Store",
                "Dept",
            ],
        )

        model_prediction_frame = (
            key_frame
            .loc[
                key_frame.index.repeat(
                    horizon
                )
            ]
            .reset_index(drop=True)
        )

        model_prediction_frame[
            "Date"
        ] = np.tile(
            target_dates.to_numpy(),
            len(key_frame),
        )

        model_prediction_frame[
            "ModelPrediction"
        ] = (
            point_forecasts
            .reshape(-1)
        )

    else:
        model_prediction_frame = (
            pd.DataFrame(
                columns=[
                    "Store",
                    "Dept",
                    "Date",
                    "ModelPrediction",
                ]
            )
        )

    result = target_rows.merge(
        model_prediction_frame,
        on=[
            "Store",
            "Dept",
            "Date",
        ],
        how="left",
        validate="many_to_one",
    )

    fallback_predictions = (
        create_fallback_predictions(
            rows=result,
            prepared_data=prepared_data,
        )
    )

    result["UsedTimesFM"] = (
        result[
            "ModelPrediction"
        ].notna()
    )

    result["Prediction"] = np.where(
        result["UsedTimesFM"],
        result["ModelPrediction"],
        fallback_predictions,
    )

    result = (
        result
        .sort_values(
            "_original_order"
        )
        .drop(
            columns=[
                "_original_order",
            ]
        )
        .reset_index(drop=True)
    )

    if result[
        "Prediction"
    ].isna().any():
        raise ValueError(
            "Final predictions contain NaN."
        )

    if not np.isfinite(
        result[
            "Prediction"
        ].to_numpy()
    ).all():
        raise ValueError(
            "Final predictions contain "
            "non-finite values."
        )

    metadata = {
        "predict_seconds": (
            predict_seconds
        ),
        "timesfm_coverage": float(
            result[
                "UsedTimesFM"
            ].mean()
        ),
        "eligible_series": len(
            eligible_keys
        ),
        "target_series": int(
            target_rows[
                ["Store", "Dept"]
            ]
            .drop_duplicates()
            .shape[0]
        ),
    }

    return result, metadata

## Load pretrained TimesFM

In [8]:
model_load_started = (
    time.perf_counter()
)

timesfm_model = (
    timesfm
    .TimesFM_2p5_200M_torch
    .from_pretrained(
        MODEL_ID
    )
)

timesfm_model.compile(
    timesfm.ForecastConfig(
        max_context=MAX_CONTEXT,
        max_horizon=64,
        normalize_inputs=True,
        per_core_batch_size=(
            PER_CORE_BATCH_SIZE
        ),
        use_continuous_quantile_head=(
            False
        ),
        force_flip_invariance=True,
        infer_is_positive=False,
        fix_quantile_crossing=False,
        return_backcast=False,
    )
)

model_load_seconds = (
    time.perf_counter()
    - model_load_started
)

print(
    "TimesFM loaded successfully."
)

print(
    "Model load seconds:",
    f"{model_load_seconds:.2f}",
)

TimesFM loaded successfully.
Model load seconds: 105.33


## WANDB logging

In [9]:
run = wandb.init(
    entity=ENTITY,
    project=WANDB_PROJECT,
    name=RUN_NAME,
    job_type="evaluation",
    config={
        "architecture": "TimesFM",
        "model_version": (
            MODEL_VERSION
        ),
        "pretrained_checkpoint": (
            MODEL_ID
        ),
        "training_mode": (
            TRAINING_MODE
        ),
        "parameter_updates": 0,
        "fine_tuning": False,
        "max_context": (
            MAX_CONTEXT
        ),
        "max_horizon": 64,
        "cv_folds": CV_FOLDS,
        "cv_horizon": (
            CV_HORIZON
        ),
        "holdout_horizon": (
            HOLDOUT_HORIZON
        ),
        "min_context_points": (
            MIN_CONTEXT_POINTS
        ),
        "min_context_coverage": (
            MIN_CONTEXT_COVERAGE
        ),
        "forecast_chunk_size": (
            FORECAST_CHUNK_SIZE
        ),
        "per_core_batch_size": (
            PER_CORE_BATCH_SIZE
        ),
        "normalize_inputs": True,
        "infer_is_positive": False,
        "seed": SEED,
    },
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/xizusha/.netrc.
wandb: Currently logged in as: lkhiz23 (lkhiz23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run e7d31a67
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /home/xizusha/Documents/ML/walmart-store-sales-forecasting/notebooks/wandb/run-20260712_045639-e7d31a67
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run timesfm__evaluation__zero_shot_target_only_v1__s42
wandb: ⭐️ View project at https://wandb.ai/lkhiz23-free-university-of-tbilisi-/walmart-store-sales-forecasting
wandb: 🚀 View run at https://wandb.ai/lkhiz23-free-university-of-tbilisi-/walmart-store-sales-forecasting/runs/e7d31a67


In [10]:
cv_results = []

for split in cv_splits:
    fold_number = split["fold"]

    fold_train = train[
        train["Date"].isin(
            split[
                "train_dates"
            ]
        )
    ].copy()

    fold_validation = train[
        train["Date"].isin(
            split[
                "validation_dates"
            ]
        )
    ].copy()

    fold_predictions, fold_metadata = (
        forecast_target_rows(
            model=timesfm_model,
            training_rows=fold_train,
            target_rows=fold_validation,
            horizon=CV_HORIZON,
        )
    )

    fold_metrics = calculate_metrics(
        fold_predictions
    )

    fold_result = {
        "fold": fold_number,
        "train_end": (
            fold_train[
                "Date"
            ].max()
        ),
        "validation_start": (
            fold_validation[
                "Date"
            ].min()
        ),
        "validation_end": (
            fold_validation[
                "Date"
            ].max()
        ),
        **fold_metrics,
        **fold_metadata,
    }

    cv_results.append(
        fold_result
    )

    run.log(
        {
            f"cv/fold_{fold_number}_wmae": (
                fold_metrics["wmae"]
            ),
            f"cv/fold_{fold_number}_mae": (
                fold_metrics["mae"]
            ),
            f"cv/fold_{fold_number}_rmse": (
                fold_metrics["rmse"]
            ),
            f"cv/fold_{fold_number}_coverage": (
                fold_metadata[
                    "timesfm_coverage"
                ]
            ),
            f"cv/fold_{fold_number}_predict_seconds": (
                fold_metadata[
                    "predict_seconds"
                ]
            ),
        }
    )

    print(
        f"Fold {fold_number}:",
        f"WMAE={fold_metrics['wmae']:.4f},",
        f"coverage="
        f"{fold_metadata['timesfm_coverage']:.2%}",
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


cv_results_frame = pd.DataFrame(
    cv_results
)

cv_mean_wmae = float(
    cv_results_frame[
        "wmae"
    ].mean()
)

cv_std_wmae = float(
    cv_results_frame[
        "wmae"
    ].std(ddof=0)
)

print(
    "\nCV mean WMAE:",
    cv_mean_wmae,
)

print(
    "CV std WMAE:",
    cv_std_wmae,
)

cv_results_frame

Fold 1: WMAE=1672.5714, coverage=96.44%
Fold 2: WMAE=1637.6538, coverage=95.59%
Fold 3: WMAE=4014.8409, coverage=94.74%

CV mean WMAE: 2441.6887078775558
CV std WMAE: 1112.4779413500166


,fold,train_end,validation_start,validation_end,wmae,mae,rmse,holiday_mae,non_holiday_mae,predict_seconds,timesfm_coverage,eligible_series,target_series
0,1,2011-04-29,2011-05-06,2011-07-29,1672.571412,1672.571412,3479.008473,NaN,1672.571412,10.563214,0.964364,2855,3104
1,2,2011-07-29,2011-08-05,2011-10-28,1637.653778,1592.809936,3549.538589,1782.986939,1576.913486,10.377984,0.955875,2840,3133
2,3,2011-10-28,2011-11-04,2012-01-27,4014.840934,3247.781112,11436.253644,5249.916754,2879.780102,10.386036,0.947394,2842,3167


In [11]:
holdout_train = train[
    train["Date"].isin(
        development_dates
    )
].copy()

holdout_validation = train[
    train["Date"].isin(
        holdout_dates
    )
].copy()

holdout_predictions, holdout_metadata = (
    forecast_target_rows(
        model=timesfm_model,
        training_rows=holdout_train,
        target_rows=holdout_validation,
        horizon=HOLDOUT_HORIZON,
    )
)

holdout_metrics = calculate_metrics(
    holdout_predictions
)

print(
    "Holdout WMAE:",
    holdout_metrics["wmae"],
)

print(
    "Holdout MAE:",
    holdout_metrics["mae"],
)

print(
    "Holdout RMSE:",
    holdout_metrics["rmse"],
)

print(
    "TimesFM coverage:",
    f"{holdout_metadata['timesfm_coverage']:.2%}",
)

print(
    "Prediction seconds:",
    holdout_metadata[
        "predict_seconds"
    ],
)

Holdout WMAE: 1706.6178810280167
Holdout MAE: 1647.8244552779358
Holdout RMSE: 3486.8750359242304
TimesFM coverage: 95.28%
Prediction seconds: 10.431863428000042


## Save results

In [12]:
final_result = pd.DataFrame(
    [
        {
            "model": "TimesFM",
            "model_version": (
                MODEL_VERSION
            ),
            "checkpoint": MODEL_ID,
            "training_mode": (
                TRAINING_MODE
            ),
            "parameter_updates": 0,
            "cv_mean_wmae": (
                cv_mean_wmae
            ),
            "cv_std_wmae": (
                cv_std_wmae
            ),
            "holdout_wmae": (
                holdout_metrics[
                    "wmae"
                ]
            ),
            "holdout_mae": (
                holdout_metrics[
                    "mae"
                ]
            ),
            "holdout_rmse": (
                holdout_metrics[
                    "rmse"
                ]
            ),
            "holiday_mae": (
                holdout_metrics[
                    "holiday_mae"
                ]
            ),
            "non_holiday_mae": (
                holdout_metrics[
                    "non_holiday_mae"
                ]
            ),
            "model_load_seconds": (
                model_load_seconds
            ),
            "holdout_predict_seconds": (
                holdout_metadata[
                    "predict_seconds"
                ]
            ),
            "holdout_coverage": (
                holdout_metadata[
                    "timesfm_coverage"
                ]
            ),
        }
    ]
)

result_path = (
    TABLES_DIR
    / "timesfm_final.csv"
)

final_result.to_csv(
    result_path,
    index=False,
)

run.log(
    {
        "results/final": (
            wandb.Table(
                dataframe=final_result
            )
        ),
        "results/cv_folds": (
            wandb.Table(
                dataframe=cv_results_frame
            )
        ),
    }
)

run.summary[
    "cv_mean_wmae"
] = cv_mean_wmae

run.summary[
    "cv_std_wmae"
] = cv_std_wmae

run.summary[
    "holdout_wmae"
] = holdout_metrics["wmae"]

run.summary[
    "holdout_mae"
] = holdout_metrics["mae"]

run.summary[
    "holdout_rmse"
] = holdout_metrics["rmse"]

run.summary[
    "holdout_coverage"
] = holdout_metadata[
    "timesfm_coverage"
]

print(
    "Saved:",
    result_path,
)

final_result

Saved: /home/xizusha/Documents/ML/walmart-store-sales-forecasting/reports/tables/timesfm_final.csv


,model,model_version,checkpoint,training_mode,parameter_updates,cv_mean_wmae,cv_std_wmae,holdout_wmae,holdout_mae,holdout_rmse,holiday_mae,non_holiday_mae,model_load_seconds,holdout_predict_seconds,holdout_coverage
0,TimesFM,2.5,google/timesfm-2.5-200m-pytorch,zero_shot,0,2441.688708,1112.477941,1706.617881,1647.824455,3486.875036,1991.342805,1629.125721,105.328669,10.431863,0.952807


## Kaggle test prediction

In [13]:
test_dates = pd.DatetimeIndex(
    test["Date"]
    .drop_duplicates()
    .sort_values()
)

if len(test_dates) != 39:
    raise ValueError(
        "Expected 39 Kaggle test weeks, "
        f"found {len(test_dates)}."
    )

test_predictions, test_metadata = (
    forecast_target_rows(
        model=timesfm_model,
        training_rows=train,
        target_rows=test,
        horizon=len(test_dates),
    )
)

print(
    "Test rows:",
    len(test_predictions),
)

print(
    "TimesFM coverage:",
    f"{test_metadata['timesfm_coverage']:.2%}",
)

print(
    "Prediction seconds:",
    test_metadata[
        "predict_seconds"
    ],
)

test_predictions[
    "Prediction"
].describe()

Test rows: 115064
TimesFM coverage: 95.76%
Prediction seconds: 10.471142375


count    115064.000000
mean      16055.611328
std       22381.037109
min       -1985.927734
25%        2108.475342
50%        7740.485596
75%       20290.233887
max      216210.468750
Name: Prediction, dtype: float64

## Build submission

In [14]:
submission_ids = (
    test["Store"]
    .astype(int)
    .astype(str)
    + "_"
    + test["Dept"]
    .astype(int)
    .astype(str)
    + "_"
    + test["Date"]
    .dt.strftime(
        "%Y-%m-%d"
    )
)

submission = pd.DataFrame(
    {
        "Id": submission_ids,
        "Weekly_Sales": (
            test_predictions[
                "Prediction"
            ]
            .to_numpy(dtype=float)
        ),
    }
)

sample_submission_candidates = [
    RAW_DATA_DIR
    / "sampleSubmission.csv",
    RAW_DATA_DIR
    / "sample_submission.csv",
]

sample_submission_path = next(
    (
        path
        for path
        in sample_submission_candidates
        if path.exists()
    ),
    None,
)

if sample_submission_path is not None:
    sample_submission = pd.read_csv(
        sample_submission_path
    )

    if (
        len(sample_submission)
        != len(submission)
    ):
        raise ValueError(
            "Submission row count differs "
            "from sample submission."
        )

    if not (
        submission["Id"]
        .astype(str)
        .reset_index(drop=True)
        .equals(
            sample_submission[
                "Id"
            ]
            .astype(str)
            .reset_index(drop=True)
        )
    ):
        raise ValueError(
            "Generated submission IDs "
            "do not match sample order."
        )

    submission["Id"] = (
        sample_submission[
            "Id"
        ].to_numpy()
    )

if submission["Id"].duplicated().any():
    raise ValueError(
        "Duplicate submission IDs."
    )

if submission[
    "Weekly_Sales"
].isna().any():
    raise ValueError(
        "Submission contains NaN."
    )

if not np.isfinite(
    submission[
        "Weekly_Sales"
    ].to_numpy()
).all():
    raise ValueError(
        "Submission contains "
        "non-finite predictions."
    )

submission_path = (
    SUBMISSIONS_DIR
    / "timesfm_submission.csv"
)

submission.to_csv(
    submission_path,
    index=False,
)

print(
    "Submission saved:",
    submission_path,
)

print(
    "Rows:",
    len(submission),
)

submission.head()

Submission saved: /home/xizusha/Documents/ML/walmart-store-sales-forecasting/reports/submissions/timesfm_submission.csv
Rows: 115064


,Id,Weekly_Sales
0,1_1_2012-11-02,29628.597656
1,1_1_2012-11-09,24430.816406
2,1_1_2012-11-16,21383.857422
3,1_1_2012-11-23,21034.039062
4,1_1_2012-11-30,25150.439453
